# Info sobre outputs
Esse arquivo adiciona as colunas de limites das tabelas de referência no geodataframe de estações, de acordo com os valores de ADT de cada linha.

**Exemplo subset_co:**
['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'REP_ESPACIAL', 'FINALIDADE',
       'STATUS', 'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR',
       'FONTE', 'CERTIFICACAO', 'COD_UF_IBGE', 'ANOS_MONITORADOS',
       'BASE_DADOS', 'ELEVACAO', 'geometry', 'EPSG', 'osm_id_1k',
       'average_daily_vehicle_count_1k', 'distance_1k', 'osm_id_10k',
       'average_daily_vehicle_count_10k', 'distance_10k', 'osm_id_20k',
       'average_daily_vehicle_count_20k', 'distance_20k', 'osm_id_30k',
       'average_daily_vehicle_count_30k', 'distance_30k', 'osm_id_40k',
       'average_daily_vehicle_count_40k', 'distance_40k', 'osm_id_50k',
       'average_daily_vehicle_count_50k', 'distance_50k', 'osm_id_60k',
       'average_daily_vehicle_count_60k', 'distance_60k', 'Razão Social',
       'industry_geom', 'distance_to_industry' 
       
       + adiciona 
       
'micro_min_1k', 'micro_max_1k',
       'bairro_min_1k', 'bairro_max_1k', 'micro_min_10k', 'micro_max_10k',
       'bairro_min_10k', 'bairro_max_10k', 'micro_min_20k', 'micro_max_20k',
       'bairro_min_20k', 'bairro_max_20k', 'micro_min_30k', 'micro_max_30k',
       'bairro_min_30k', 'bairro_max_30k', 'micro_min_40k', 'micro_max_40k',
       'bairro_min_40k', 'bairro_max_40k', 'micro_min_50k', 'micro_max_50k',
       'bairro_min_50k', 'bairro_max_50k', 'micro_min_60k', 'micro_max_60k',
       'bairro_min_60k', 'bairro_max_60k']

# Importando variáveis

In [ ]:
from 03_prep_tabelas_ref.ipynb import interpolated_dict

## 7.3 Adicionando ao gdf de estradas os limites das tabelas de referência interpoladas

In [15]:
# Iterando sobre os poluentes e seus subconjuntos
for poll, subset in pollutant_subsets.items():
    
    # Iterando sobre os valores de ADT da ref_table para cada poluente
    for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
        col_name = f'average_daily_vehicle_count_{adt_band}k'
        
        # Registrando valores de ADT sem vias para cada poluente
        if col_name not in subset.columns:
            print(f"[AVISO] '{col_name}' não existe no subset_{poll}")
            continue
        
        # Obtendo a tabela do lado direito (para o merge)
        interp_df = interpolated_dict[poll].copy()
        
        # Renomeando a primeira coluna, para que a função merge não adicione sufixo
        suffix = f"_{adt_band}k"
        interp_df = interp_df.rename(columns={
            col: f"{col}{suffix}" 
            for col
            in interp_df.columns
            if col != 'avg_adt'
        })
        
        # Mesclando colunas com limites de distância para cada ADT e classe representativa
        pollutant_subsets[poll] = pollutant_subsets[poll].merge(
            right= interp_df,
            how='left',
            left_on= col_name,
            right_on='avg_adt',
            suffixes=(None, f"_{adt_band}k")
            )
        
        # Removendo colunas 'avg_adt_{}k' adicionadas anteriormente
        if (idx != 0) and (f'avg_adt_{adt_band}k' 
                           in pollutant_subsets[poll].columns):
            pollutant_subsets[poll].drop(columns=[f'avg_adt_{adt_band}k'],
                                         inplace=True)
    
    # Remove a primeira coluna 'avg_adt' adicionada
    pollutant_subsets[poll].drop(columns=['avg_adt'], inplace=True)
        
    # Preenchendo valores nulos com np.inf (somente colunas 
    # {micro/meso/bairro/urb}_max podem ter nulos)
    cols = list(pollutant_subsets[poll].filter(like='k').columns)
    pollutant_subsets[poll].loc[:,cols] = (
        pollutant_subsets[poll]
        .loc[:,cols]
        .astype(float)
        .fillna(np.inf)
    )
        
del suffix


In [16]:
pollutant_subsets['co'].columns

Index(['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'REP_ESPACIAL', 'FINALIDADE',
       'STATUS', 'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR',
       'FONTE', 'CERTIFICACAO', 'COD_UF_IBGE', 'ANOS_MONITORADOS',
       'BASE_DADOS', 'ELEVACAO', 'geometry', 'EPSG', 'osm_id_1k',
       'average_daily_vehicle_count_1k', 'distance_1k', 'osm_id_10k',
       'average_daily_vehicle_count_10k', 'distance_10k', 'osm_id_20k',
       'average_daily_vehicle_count_20k', 'distance_20k', 'osm_id_30k',
       'average_daily_vehicle_count_30k', 'distance_30k', 'osm_id_40k',
       'average_daily_vehicle_count_40k', 'distance_40k', 'osm_id_50k',
       'average_daily_vehicle_count_50k', 'distance_50k', 'osm_id_60k',
       'average_daily_vehicle_count_60k', 'distance_60k', 'Razão S